# CRISPRitz Off-Target Search

![CRISPRitz Off-Target Search](https://proto-bio.github.io/proto-assets/images/tool/crispritz/hero.png)

This notebook demonstrates `run_crispritz_search`, which enumerates every site in a local reference genome that a guide could bind, allowing both mismatches and DNA/RNA bulges. [CRISPRitz](https://github.com/pinellolab/CRISPRitz) builds a PAM-aware index over the reference once, then searches every guide against it.

The search runs locally: it reads a reference FASTA you supply and keeps its index in a local cache, so it does not run on hosted devices.

In [1]:
from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.sequence_alignment.crispritz import (
    CRISPRitzSearchConfig,
    CRISPRitzSearchInput,
    run_crispritz_search,
)

## API Reference

In [2]:
display_api_reference("crispritz-search", "input", "run_crispritz_search")

**Input** — `CRISPRitzSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>guides</code> | <code>list[str]</code> | required | PAM-free guide DNA/RNA protospacers; all entries must have the same length |
| <code>reference_fasta</code> | <code>str</code> | required | Local uncompressed FASTA file or directory of .fa/.fasta/.fna files |
| <code>reference_id</code> | <code>str &#124; None</code> | <code>None</code> | Optional versioned assembly or accession identifier retained as provenance |

In [3]:
display_api_reference("crispritz-search", "config", "run_crispritz_search")

**Config** — `CRISPRitzSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pam</code> | <code>str</code> | <code>'NGG'</code> | IUPAC PAM motif without protospacer bases |
| <code>pam_side</code> | <code>Literal['three_prime', 'five_prime']</code> | <code>'three_prime'</code> | PAM position relative to the protospacer |
| <code>max_mismatches</code> | <code>int</code> | <code>4</code> | Inclusive substitution limit per hit |
| <code>max_dna_bulges</code> | <code>int</code> | <code>1</code> | Inclusive DNA bulge-length limit per hit |
| <code>max_rna_bulges</code> | <code>int</code> | <code>1</code> | Inclusive RNA bulge-length limit per hit |
| <code>max_total_edits</code> | <code>int &#124; None</code> | <code>None</code> | Optional inclusive cap on mismatches plus bulge length |
| <code>threads</code> | <code>int</code> | <code>4</code> | CPU threads for indexing and search |
| <code>index_cache_dir</code> | <code>str &#124; None</code> | <code>None</code> | Persistent checksum-keyed CRISPRitz index cache; defaults under PROTO_HOME |

In [4]:
display_api_reference("crispritz-search", "output", "run_crispritz_search")

**Output** — `CRISPRitzSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>hits</code> | <code>list[CRISPRitzHit]</code> | <code>[]</code> | Enumerated target sites |
| <code>guides</code> | <code>list[str]</code> | required | Normalized PAM-free queried protospacers |
| <code>reference_id</code> | <code>str &#124; None</code> | <code>None</code> | Reference accession |
| <code>reference_sha256</code> | <code>str</code> | required | Combined reference FASTA checksum |
| <code>index_reused</code> | <code>bool</code> | required | Whether an existing completed index was reused |
| <code>crispritz_version</code> | <code>str</code> | required | Pinned upstream version |
| <code>pam</code> | <code>str</code> | required | IUPAC PAM motif |
| <code>pam_side</code> | <code>Literal['three_prime', 'five_prime']</code> | required | PAM orientation |
| <code>max_mismatches</code> | <code>int</code> | required | Search substitution limit |
| <code>max_dna_bulges</code> | <code>int</code> | required | Search DNA-bulge limit |
| <code>max_rna_bulges</code> | <code>int</code> | required | Search RNA-bulge limit |
| <code>max_total_edits</code> | <code>int &#124; None</code> | required | Optional total-edit limit |

**`CRISPRitzHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>guide</code> | <code>str</code> | required | Normalized PAM-free guide protospacer |
| <code>bulge_type</code> | <code>Literal['none', 'dna', 'rna', 'dna_rna']</code> | required | Normalized alignment indel class |
| <code>crispritz_bulge_type</code> | <code>str</code> | required | Unmodified upstream X, DNA, RNA, or RNA,DNA label |
| <code>guide_alignment</code> | <code>str</code> | required | CRISPRitz crRNA alignment |
| <code>target_alignment</code> | <code>str</code> | required | CRISPRitz genomic alignment |
| <code>chromosome</code> | <code>str</code> | required | Reference FASTA record identifier |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive genomic start of the aligned target |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive genomic end of the aligned target |
| <code>strand</code> | <code>Literal['+', '-']</code> | required | Genomic target strand |
| <code>mismatches</code> | <code>int</code> | required | Substitution count excluding bulges |
| <code>bulge_size</code> | <code>int</code> | required | Bulge length |
| <code>total_edits</code> | <code>int</code> | required | Mismatches plus bulge length |

## Search a Local Reference

Guides are PAM-free protospacers of a single length; the PAM comes from the config, written in nuclease orientation. `reference_fasta` is an uncompressed FASTA file or a directory of them, such as the genome inside an `ncbi-reference-bundle` result.

In [5]:
from pathlib import Path

reference = Path("../example_reference.fasta")
result = run_crispritz_search(
    CRISPRitzSearchInput(
        guides=["GACTGCATTGCATGACTGCA"],
        reference_fasta=str(reference),
        reference_id="synthetic-crispritz-example-v1",
    ),
    CRISPRitzSearchConfig(max_mismatches=4, max_dna_bulges=1, max_rna_bulges=1),
)

print(f"{len(result.hits)} hits, index reused: {result.index_reused}")
for hit in result.hits:
    print(f"{hit.chromosome}:{hit.start}-{hit.end} {hit.strand} {hit.bulge_type} mm={hit.mismatches}")

Running run_crispritz_search [00:00]

380 hits, index reused: True
synthetic_chr:32-54 + none mm=0
synthetic_chr:5-27 + none mm=0
synthetic_chr:33-54 + rna mm=0
synthetic_chr:6-27 + rna mm=0
synthetic_chr:31-54 + dna mm=1
synthetic_chr:4-27 + dna mm=1
synthetic_chr:33-54 + rna mm=1
synthetic_chr:6-27 + rna mm=1
synthetic_chr:32-54 + dna_rna mm=0
synthetic_chr:5-27 + dna_rna mm=0
synthetic_chr:31-54 + dna mm=2
synthetic_chr:4-27 + dna mm=2
synthetic_chr:32-54 + dna_rna mm=0
synthetic_chr:5-27 + dna_rna mm=0
synthetic_chr:33-54 + rna mm=2
synthetic_chr:6-27 + rna mm=2
synthetic_chr:32-54 + dna_rna mm=1
synthetic_chr:5-27 + dna_rna mm=1
synthetic_chr:32-54 + dna_rna mm=0
synthetic_chr:5-27 + dna_rna mm=0
synthetic_chr:31-54 + dna mm=3
synthetic_chr:4-27 + dna mm=3
synthetic_chr:32-54 + dna_rna mm=1
synthetic_chr:5-27 + dna_rna mm=1
synthetic_chr:32-54 + dna_rna mm=0
synthetic_chr:5-27 + dna_rna mm=0
synthetic_chr:33-54 + rna mm=3
synthetic_chr:6-27 + rna mm=3
synthetic_chr:32-54 + dna_rna mm=2
synthetic_chr:5-27 + dna_rna mm=

## Read an Alignment

`guide_alignment` and `target_alignment` are the aligned strings CRISPRitz produced, with `-` marking a bulge, so a hit can be inspected base by base. Coordinates are 1-indexed and inclusive, like everywhere else in Proto.

In [6]:
for hit in result.hits[:3]:
    print(f"guide  {hit.guide_alignment}")
    print(f"target {hit.target_alignment}")
    print(f"       {hit.mismatches} mismatches, {hit.bulge_size}-nt {hit.bulge_type} bulge\n")

guide  GACTGCATTGCATGACTGCANNN
target GACTGCATTGCATGACTGCACGG
       0 mismatches, 0-nt none bulge

guide  GACTGCATTGCATGACTGCANNN
target GACTGCATTGCATGACTGCAAGG
       0 mismatches, 0-nt none bulge

guide  GACTGCATTGCATGACTGCANNN
target -ACTGCATTGCATGACTGCACGG
       0 mismatches, 1-nt rna bulge



## Tighten the Search

Mismatch and bulge limits are inclusive maxima, and `max_total_edits` caps mismatches plus bulge length together, which is usually the cheaper way to cut a large hit list down.

In [7]:
strict = run_crispritz_search(
    CRISPRitzSearchInput(guides=["GACTGCATTGCATGACTGCA"], reference_fasta=str(reference)),
    CRISPRitzSearchConfig(max_mismatches=2, max_dna_bulges=0, max_rna_bulges=0, max_total_edits=2),
)
print(f"{len(result.hits)} hits with the permissive settings, {len(strict.hits)} with the strict ones")

Running run_crispritz_search [00:00]

380 hits with the permissive settings, 2 with the strict ones


## Export Results

JSON keeps the search parameters and reference checksum alongside the hits; CSV flattens to one row per hit.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispritz_hits", export_path=output_dir, file_format="csv")
print(f"Off-target hits exported to {output_dir}")

Off-target hits exported to example_output
